# 1. Montar Google Drive e Importaciones

In [ ]:
# ==========================================
# 1. MONTAR GOOGLE DRIVE E INSTALAR DEPENDENCIAS
# ==========================================
from collections import defaultdict
import gc
import json
import os
import re
import shutil
from google.colab import drive
import orjson
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


# 2. Configuración de Rutas

In [ ]:
# ==========================================
# 2. CONFIGURACIÓN DE RUTAS SEGÚN TUS CARPETAS
# ==========================================
BASE_DRIVE = '/content/drive/MyDrive/TFG/data'

DIR_PEMR = os.path.join(BASE_DRIVE, 'PatientEncounterMedRequest')
DIR_VARIOS = os.path.join(BASE_DRIVE, 'Varios')
DIR_REST = os.path.join(BASE_DRIVE, 'Rest')

# Carpeta local temporal en Colab para guardar el resultado de salida sin llenar tu PC
OUTPUT_DIR = '/content/output_tfg'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'medication_request_dataset_full.jsonl')

# Recursos FHIR
PATIENT_FILE = os.path.join(DIR_PEMR, 'MimicPatient.ndjson')
MED_REQUEST_FILE = os.path.join(DIR_PEMR, 'MimicMedicationRequest.ndjson')
ENCOUNTER_FILES = [
    os.path.join(DIR_PEMR, 'MimicEncounter.ndjson'),
    os.path.join(DIR_REST, 'MimicEncounterED.ndjson')
]
CONDITION_FILES = [
    os.path.join(DIR_VARIOS, 'MimicCondition.ndjson'),
    os.path.join(DIR_REST, 'MimicConditionED.ndjson')
]
MED_FILE = os.path.join(DIR_VARIOS, 'MimicMedication.ndjson')
MED_MIX_FILE = os.path.join(
    DIR_REST, 'MimicMedicationMix.ndjson'
)
ADMIN_FILE = os.path.join(DIR_REST, 'MimicMedicationAdministration.ndjson')
PROCEDURE_FILES = [
    os.path.join(DIR_VARIOS, 'MimicProcedure.ndjson')
]

# CSVs de MIMIC-IV 2.2
CSV_MEDICOS = os.path.join(BASE_DRIVE, 'prescriptions.csv.gz')
PATIENTS_CSV_FILE = os.path.join(BASE_DRIVE, 'patients.csv')

# 3. Herramientas Base y Ajuste de Fechas

In [ ]:
def iter_ndjson(path, stats):
  """Lee archivos NDJSON línea a línea con orjson y registra errores de parseo."""
  if not os.path.exists(path):
    print(f'⚠️ Aviso: No se encontró {path}')
    return
  with open(path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
      if not line.strip():
        continue
      try:
        yield orjson.loads(line)
      except (orjson.JSONDecodeError, ValueError):
        stats['json_errors'] += 1


def dig(d, *path, default=None):
  """Acceso seguro a estructuras anidadas FHIR."""
  for k in path:
    try:
      d = d[k]
    except (KeyError, IndexError, TypeError):
      return default
  return d


def parse_primer_anio_grupo(anchor_year_group):
  """Extrae el primer año numérico de anchor_year_group (p. ej. '2008 - 2010' -> 2008)."""
  if anchor_year_group is None or pd.isna(anchor_year_group):
    return None
  m = re.search(r'(\d{4})', str(anchor_year_group))
  return int(m.group(1)) if m else None


def transformar_fecha_anio_aprox(fecha_str, anchor_year, primer_anio):
  """
    Calcula anio_aprox = anio(fecha_desplazada) - anchor_year + primer_anio
    conservando mes y día (y hora si viene en la fecha).
    Devuelve (fecha_ajustada_iso_o_str, anio_aprox).
    """
  if not fecha_str or anchor_year is None or primer_anio is None:
    return fecha_str if fecha_str else '', None
  if pd.isna(anchor_year) or pd.isna(primer_anio):
    return fecha_str if fecha_str else '', None

  dt = pd.to_datetime(str(fecha_str), errors='coerce')
  if pd.isna(dt):
    return str(fecha_str), None

  anio_aprox = int(dt.year) - int(anchor_year) + int(primer_anio)
  try:
    dt_ajustada = dt.replace(year=anio_aprox)
  except ValueError:
    # Si era 29 de febrero y el año aproximado no es bisiesto
    dt_ajustada = dt.replace(year=anio_aprox, month=2, day=28)

  # Si la cadena original no tenía hora (ej. YYYY-MM-DD), devolvemos fecha corta
  tiene_hora = ('T' in str(fecha_str)) or (':' in str(fecha_str))
  if tiene_hora:
    return dt_ajustada.isoformat(), anio_aprox
  return dt_ajustada.strftime('%Y-%m-%d'), anio_aprox


def normalizar_nombre_farmaco(nombre_raw):
  """
    Si el nombre corresponde a una mezcla separada por '--', toma el primer componente,
    pasa a minúsculas y deja espacios simples.
    """
  if not nombre_raw:
    return 'unknown'
  primer_componente = str(nombre_raw).split('--')[0]
  limpio = re.sub(r'\s+', ' ', primer_componente.strip().lower())
  return limpio if limpio else 'unknown'

# 4. Funciones de Carga en RAM (Dimensiones y Mapeo Temporal)

In [ ]:
# ==========================================
# 4. FUNCIONES DE CARGA OPTIMIZADAS (FILTRADO EN ORIGEN)
# ==========================================
def cargar_mapa_anclaje_pacientes(patients_csv_path):
  """
    Lee patients.csv (MIMIC-IV 2.2) y devuelve un diccionario indexado por subject_id (str)
    con anchor_year, primer_anio y dod.
    """
  df_pat = pd.read_csv(
      patients_csv_path,
      usecols=['subject_id', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype={'subject_id': str, 'anchor_year_group': str, 'dod': str},
  )
  mapa = {}
  for row in df_pat.itertuples(index=False):
    sid = str(row.subject_id).strip()
    primer_anio = parse_primer_anio_grupo(row.anchor_year_group)
    anchor_yr = (
        int(row.anchor_year) if pd.notna(row.anchor_year) else None
    )
    dod_val = row.dod.strip() if pd.notna(row.dod) else ''
    mapa[sid] = {
        'anchor_year': anchor_yr,
        'anchor_year_group': row.anchor_year_group if pd.notna(row.anchor_year_group) else '',
        'primer_anio': primer_anio,
        'dod_raw': dod_val,
    }
  return mapa


def cargar_pacientes(filepath, mapa_anclaje, stats):
  """
    Carga los pacientes del recurso FHIR y calcula birthDate y defunción aproximadas
    usando anchor_year y primer_anio de patients.csv.
    """
  print('⏳ Cargando pacientes y calculando fechas aproximadas...')
  diccionario = {}

  for paciente in iter_ndjson(filepath, stats):
    fhir_uuid = paciente.get('id')
    subject_id_num = None
    for ident in paciente.get('identifier', []):
      if 'mimic/identifier/patient' in str(ident.get('system', '')):
        subject_id_num = str(ident.get('value')).strip()
        break

    info_anclaje = mapa_anclaje.get(subject_id_num, {})
    anchor_yr = info_anclaje.get('anchor_year')
    primer_anio = info_anclaje.get('primer_anio')

    birth_raw = paciente.get('birthDate', '')
    birth_ajustada, _ = transformar_fecha_anio_aprox(
        birth_raw, anchor_yr, primer_anio
    )

    dod_origen = info_anclaje.get('dod_raw') or paciente.get(
        'deceasedDateTime', ''
    )
    dod_ajustada, _ = transformar_fecha_anio_aprox(
        dod_origen, anchor_yr, primer_anio
    )

    diccionario[fhir_uuid] = {
        'subject_id': subject_id_num,
        'anchor_year': anchor_yr,
        'anchor_year_group': info_anclaje.get('anchor_year_group', ''),
        'primer_anio': primer_anio,
        'patient_gender': paciente.get('gender', 'unknown'),
        'patient_birthdate': birth_ajustada,
        'patient_dod': dod_ajustada,
    }

  print(f'✅ Pacientes cargados en RAM: {len(diccionario):,}')
  return diccionario


def cargar_encounters(filepaths, pacientes_dict, stats):
  """Carga los encuentros y ajusta start y end con el anclaje del paciente."""
  print('⏳ Cargando ingresos hospitalarios...')
  diccionario = {}

  for filepath in filepaths:
    for enc in iter_ndjson(filepath, stats):
      pat_ref = dig(enc, 'subject', 'reference', default='')
      pat_id = pat_ref.rsplit('/', 1)[-1] if '/' in pat_ref else ''
      pat_info = pacientes_dict.get(pat_id)

      anchor_yr = pat_info.get('anchor_year') if pat_info else None
      primer_anio = pat_info.get('primer_anio') if pat_info else None

      start_raw = dig(enc, 'period', 'start', default='')
      end_raw = dig(enc, 'period', 'end', default='')

      start_aj, _ = transformar_fecha_anio_aprox(
          start_raw, anchor_yr, primer_anio
      )
      end_aj, _ = transformar_fecha_anio_aprox(end_raw, anchor_yr, primer_anio)

      enc_id = enc.get('id')
      diccionario[enc_id] = {
          'patient_fhir_id': pat_id,
          'encounter_class': dig(enc, 'class', 'code', default='unknown'),
          'encounter_start': start_aj,
          'encounter_end': end_aj,
          'encounter_service_type': dig(
              enc, 'serviceType', 'coding', 0, 'code', default='unknown'
          ),
      }
  print(f'✅ Encuentros cargados: {len(diccionario):,}')
  return diccionario


def cargar_diagnosticos(filepaths, stats):
  """Carga los diagnósticos agrupados por encounter_id."""
  print('⏳ Cargando diagnósticos...')
  por_enc = defaultdict(dict)
  for filepath in filepaths:
    for cond in iter_ndjson(filepath, stats):
      enc_ref = dig(cond, 'encounter', 'reference', default='')
      enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else ''
      if enc_id:
        cod = dig(cond, 'code', 'coding', 0, 'code', default='UNKNOWN')
        txt = dig(cond, 'code', 'coding', 0, 'display', default='No description')
        por_enc[enc_id][cod] = txt
  return por_enc


def cargar_procedimientos(filepaths, stats):
  """Carga los procedimientos agrupados por encounter_id."""
  print('⏳ Cargando procedimientos...')
  por_enc = defaultdict(dict)
  for filepath in filepaths:
    for proc in iter_ndjson(filepath, stats):
      enc_ref = dig(proc, 'encounter', 'reference', default='')
      if not enc_ref:
        enc_ref = proc.get('encounter', '')
        if isinstance(enc_ref, dict):
          enc_ref = enc_ref.get('reference', '')
      enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else str(enc_ref)
      if enc_id:
        cod = dig(proc, 'code', 'coding', 0, 'code', default='UNKNOWN')
        txt = dig(proc, 'code', 'coding', 0, 'display', default='No description')
        por_enc[enc_id][cod] = txt
  return por_enc


def cargar_catalogo_medicamentos(med_file, med_mix_file, stats):
  """
    Carga tanto MimicMedication (system con 'mimic-medication-name') como
    MimicMedicationMix (system con 'medication-mix') y devuelve un diccionario
    indexado por referencia ('MimicMedication/<id>', 'MimicMedicationMix/<id>')
    y también por <id> con el nombre ya normalizado.
    """
  print('⏳ Cargando catálogos MimicMedication y MimicMedicationMix...')
  catalogo = {}

  # 1. MimicMedication
  for med in iter_ndjson(med_file, stats):
    med_id = med.get('id')
    med_name = None
    for ident in med.get('identifier', []):
      if 'mimic-medication-name' in str(ident.get('system', '')):
        med_name = ident.get('value')
        break
    if not med_name:
      med_name = dig(med, 'code', 'text', default='unknown')
    norm_name = normalizar_nombre_farmaco(med_name)
    if med_id:
      catalogo[med_id] = norm_name
      catalogo[f'MimicMedication/{med_id}'] = norm_name

  # 2. MimicMedicationMix
  for mix in iter_ndjson(med_mix_file, stats):
    mix_id = mix.get('id')
    mix_name = None
    for ident in mix.get('identifier', []):
      if 'medication-mix' in str(ident.get('system', '')):
        mix_name = ident.get('value')
        break
    if not mix_name:
      mix_name = dig(mix, 'code', 'text', default='unknown')
    norm_name = normalizar_nombre_farmaco(mix_name)
    if mix_id:
      catalogo[mix_id] = norm_name
      catalogo[f'MimicMedicationMix/{mix_id}'] = norm_name

  print(f'✅ Entradas normalizadas en catálogo de fármacos: {len(catalogo):,}')
  return catalogo


def cargar_administraciones(filepath, stats):
  """Carga los IDs de peticiones con registro de administración."""
  print('⏳ Cargando registros de administración...')
  administraciones = set()
  for admin in iter_ndjson(filepath, stats):
    req_ref = dig(admin, 'request', 'reference', default='')
    req_id = req_ref.rsplit('/', 1)[-1] if '/' in req_ref else ''
    if req_id:
      administraciones.add(req_id)
  return administraciones

# 5. Carga de Mapas y Dimensiones en RAM

In [ ]:
# ==============================================================================
# CELDA A: CARGA DE MAPAS (MIMIC-IV 2.2) Y DIMENSIONES EN RAM
# ==============================================================================

print("=" * 80)
print(" [A] CARGANDO MAPAS Y DIMENSIONES (MIMIC-IV 2.2)")
print("=" * 80)

# 1. Mapa de médicos desde prescriptions.csv.gz (MIMIC-IV 2.2)
print("\n⏳ [1/3] Cargando mapa de médicos (pharmacy_id -> order_provider_id)...")
if not os.path.exists(CSV_MEDICOS):
  raise FileNotFoundError(f"❌ No se encuentra el archivo CSV en {CSV_MEDICOS}.")

df_med = pd.read_csv(
    CSV_MEDICOS,
    usecols=["pharmacy_id", "order_provider_id"],
    dtype={"pharmacy_id": str, "order_provider_id": str},
)
df_med = df_med.dropna(subset=["pharmacy_id", "order_provider_id"])
df_med["pharmacy_id"] = df_med["pharmacy_id"].str.strip()
df_med["order_provider_id"] = df_med["order_provider_id"].str.strip()
df_med = df_med.drop_duplicates(subset=["pharmacy_id"], keep="first")
mapa_medicos = dict(zip(df_med["pharmacy_id"], df_med["order_provider_id"]))
del df_med
print(f"✅ Mapa de médicos cargado: {len(mapa_medicos):,} pharmacy_id únicos.")

# 2. Mapa de anclaje de pacientes desde patients.csv (MIMIC-IV 2.2)
print(
    "\n⏳ [2/3] Cargando anchor_year y anchor_year_group desde patients.csv..."
)
if not os.path.exists(PATIENTS_CSV_FILE):
  raise FileNotFoundError(
      f"❌ No se encuentra patients.csv en {PATIENTS_CSV_FILE}."
  )

mapa_anclaje = cargar_mapa_anclaje_pacientes(PATIENTS_CSV_FILE)
print(f"✅ Datos de anclaje cargados: {len(mapa_anclaje):,} pacientes.")

# 3. Dimensiones FHIR
stats = defaultdict(int)
print("\n⏳ [3/3] Cargando dimensiones FHIR en RAM...")
pacientes = cargar_pacientes(PATIENT_FILE, mapa_anclaje, stats)
encounters = cargar_encounters(ENCOUNTER_FILES, pacientes, stats)
diagnosticos = cargar_diagnosticos(CONDITION_FILES, stats)
procedimientos = cargar_procedimientos(PROCEDURE_FILES, stats)
catalogo_medicamentos = cargar_catalogo_medicamentos(
    MED_FILE, MED_MIX_FILE, stats
)

gc.collect()
print("\n✅ Celda A lista. Dimensiones cargadas en memoria.")

 [A] CARGANDO MAPAS Y DIMENSIONES (MIMIC-IV 2.2)

⏳ [1/3] Cargando mapa de médicos (pharmacy_id -> order_provider_id)...
✅ Mapa de médicos cargado: 13,478,956 pharmacy_id únicos.

⏳ [2/3] Cargando anchor_year y anchor_year_group desde patients.csv...
✅ Datos de anclaje cargados: 299,712 pacientes.

⏳ [3/3] Cargando dimensiones FHIR en RAM...
⏳ Cargando pacientes y calculando fechas aproximadas...
✅ Pacientes cargados en RAM: 299,712
⏳ Cargando ingresos hospitalarios...
✅ Encuentros cargados: 856,318
⏳ Cargando diagnósticos...
⏳ Cargando procedimientos...
⏳ Cargando catálogos MimicMedication y MimicMedicationMix...
✅ Entradas normalizadas en catálogo de fármacos: 52,054

✅ Celda A lista. Dimensiones cargadas en memoria.


POR SI SE DESCONECTA

In [ ]:
# ==============================================================================
# CELDA [A] CON CACHÉ EN GOOGLE DRIVE (CARGA RÁPIDA SI SE DESCONECTA)
# ==============================================================================

import os, gc, pickle, pandas as pd
from collections import defaultdict

CACHE_DIMENSIONES_FILE = os.path.join(BASE_DRIVE, "dimensiones_mimic22_cache.pkl")
FORZAR_RECARGA = False  # Pon True solo si cambias la lógica de carga o los CSV/NDJSON

print("=" * 80)
print(" [A] DIMENSIONES MIMIC-IV 2.2 (CON CACHÉ EN DRIVE)")
print("=" * 80)

if os.path.exists(CACHE_DIMENSIONES_FILE) and not FORZAR_RECARGA:
    print(f"\n⚡ Caché encontrada en Drive: {CACHE_DIMENSIONES_FILE}")
    print("⏳ Cargando dimensiones directamente desde caché...")
    with open(CACHE_DIMENSIONES_FILE, "rb") as f_cache:
        cache_data = pickle.load(f_cache)

    mapa_medicos = cache_data["mapa_medicos"]
    mapa_anclaje = cache_data["mapa_anclaje"]
    pacientes = cache_data["pacientes"]
    encounters = cache_data["encounters"]
    diagnosticos = cache_data["diagnosticos"]
    procedimientos = cache_data["procedimientos"]
    catalogo_medicamentos = cache_data["catalogo_medicamentos"]
    stats = defaultdict(int, cache_data.get("stats", {}))

    print("✅ Dimensiones cargadas desde caché al instante:")
    print(f"   - Médicos (pharmacy_id): {len(mapa_medicos):,}")
    print(f"   - Pacientes: {len(pacientes):,}")
    print(f"   - Encuentros: {len(encounters):,}")
    print(f"   - Fármacos/mezclas: {len(catalogo_medicamentos):,}")

else:
    print("\n⏳ [1/3] Cargando mapa de médicos (pharmacy_id -> order_provider_id)...")
    if not os.path.exists(CSV_MEDICOS):
        raise FileNotFoundError(f"❌ No se encuentra el archivo CSV en {CSV_MEDICOS}.")

    df_med = pd.read_csv(
        CSV_MEDICOS,
        usecols=["pharmacy_id", "order_provider_id"],
        dtype={"pharmacy_id": str, "order_provider_id": str},
    )
    df_med = df_med.dropna(subset=["pharmacy_id", "order_provider_id"])
    df_med["pharmacy_id"] = df_med["pharmacy_id"].str.strip()
    df_med["order_provider_id"] = df_med["order_provider_id"].str.strip()
    df_med = df_med.drop_duplicates(subset=["pharmacy_id"], keep="first")
    mapa_medicos = dict(zip(df_med["pharmacy_id"], df_med["order_provider_id"]))
    del df_med
    print(f"✅ Mapa de médicos cargado: {len(mapa_medicos):,} pharmacy_id únicos.")

    print("\n⏳ [2/3] Cargando anchor_year y anchor_year_group desde patients.csv...")
    if not os.path.exists(PATIENTS_CSV_FILE):
        raise FileNotFoundError(f"❌ No se encuentra patients.csv en {PATIENTS_CSV_FILE}.")

    mapa_anclaje = cargar_mapa_anclaje_pacientes(PATIENTS_CSV_FILE)
    print(f"✅ Datos de anclaje cargados: {len(mapa_anclaje):,} pacientes.")

    stats = defaultdict(int)
    print("\n⏳ [3/3] Cargando dimensiones FHIR en RAM...")
    pacientes = cargar_pacientes(PATIENT_FILE, mapa_anclaje, stats)
    encounters = cargar_encounters(ENCOUNTER_FILES, pacientes, stats)
    diagnosticos = cargar_diagnosticos(CONDITION_FILES, stats)
    procedimientos = cargar_procedimientos(PROCEDURE_FILES, stats)
    catalogo_medicamentos = cargar_catalogo_medicamentos(MED_FILE, MED_MIX_FILE, stats)

    print(f"\n💾 Guardando caché de dimensiones en Drive: {CACHE_DIMENSIONES_FILE}...")
    with open(CACHE_DIMENSIONES_FILE, "wb") as f_cache:
        pickle.dump(
            {
                "mapa_medicos": mapa_medicos,
                "mapa_anclaje": mapa_anclaje,
                "pacientes": pacientes,
                "encounters": encounters,
                "diagnosticos": diagnosticos,
                "procedimientos": procedimientos,
                "catalogo_medicamentos": catalogo_medicamentos,
                "stats": dict(stats),
            },
            f_cache,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    print("✅ Caché guardada correctamente en Google Drive.")

gc.collect()

 [A] DIMENSIONES MIMIC-IV 2.2 (CON CACHÉ EN DRIVE)

⚡ Caché encontrada en Drive: /content/drive/MyDrive/TFG/data/dimensiones_mimic22_cache.pkl
⏳ Cargando dimensiones directamente desde caché...
✅ Dimensiones cargadas desde caché al instante:
   - Médicos (pharmacy_id): 13,478,956
   - Pacientes: 299,712
   - Encuentros: 856,318
   - Fármacos/mezclas: 52,054


0

# 6. Procesamiento de MedicationRequest y exportación (Train 2008 / Eval 2009)

In [ ]:
# ==============================================================================
# CELDA [B] UNIFICADA Y RÁPIDA: MEDICATIONREQUEST -> TRAIN (2008) & EVAL (2009 Q1)
# ==============================================================================

import os
import shutil
import orjson

print("=" * 80)
print(" 🚀 [B] PROCESANDO MEDICATIONREQUEST (TRAIN 2008 / EVAL ENE-MAR 2009)")
print("=" * 80)

OUT_TRAIN_LOCAL = "/content/medication_request_2008.jsonl"
OUT_EVAL_LOCAL = "/content/medication_request_2009_q1.jsonl"
TMP_TRAIN_LOCAL = OUT_TRAIN_LOCAL + ".tmp"
TMP_EVAL_LOCAL = OUT_EVAL_LOCAL + ".tmp"

DRIVE_TRAIN_FINAL = os.path.join(BASE_DRIVE, "medication_request_2008.jsonl")
DRIVE_EVAL_FINAL = os.path.join(BASE_DRIVE, "medication_request_2009_q1.jsonl")


def transformar_fecha_rapida(fecha_str, anchor_yr, primer_anio):
  """Slicing directo YYYY-MM-DD... sin sobrecarga de pandas."""
  if not fecha_str or len(fecha_str) < 10:
    return None, None, None
  try:
    anio_desp = int(fecha_str[:4])
    mes = int(fecha_str[5:7])
    dia = int(fecha_str[8:10])
  except ValueError:
    return None, None, None

  anio_aprox = anio_desp - anchor_yr + primer_anio
  if mes == 2 and dia == 29 and (anio_aprox % 4 != 0 or (anio_aprox % 100 == 0 and anio_aprox % 400 != 0)):
    fecha_aj = f"{anio_aprox:04d}-02-28{fecha_str[10:]}"
  else:
    fecha_aj = f"{anio_aprox:04d}{fecha_str[4:]}"
  return fecha_aj, anio_aprox, mes


cont_total = 0
cont_train = 0
cont_eval = 0

with (
    open(TMP_TRAIN_LOCAL, "wb") as f_train,
    open(TMP_EVAL_LOCAL, "wb") as f_eval,
):

  for receta in iter_ndjson(MED_REQUEST_FILE, stats):
    cont_total += 1
    stats["total_recetas_procesadas"] += 1

    try:
      pat_ref = dig(receta, "subject", "reference", default="")
      patient_id = pat_ref.rsplit("/", 1)[-1] if "/" in pat_ref else ""

      datos_pac = pacientes.get(patient_id)
      if not datos_pac:
        stats["paciente_no_encontrado"] += 1
        continue

      anchor_yr = datos_pac["anchor_year"]
      primer_anio = datos_pac["primer_anio"]
      if anchor_yr is None or primer_anio is None:
        stats["sin_anclaje_valido"] += 1
        continue

      authored_raw = receta.get("authoredOn")
      if not authored_raw:
        stats["sin_authored_on"] += 1
        continue

      authored_aj, anio_aprox, mes_auth = transformar_fecha_rapida(
          authored_raw, anchor_yr, primer_anio
      )
      if anio_aprox is None:
        stats["fecha_invalida"] += 1
        continue

      if anio_aprox == 2008:
        split = "train"
      elif anio_aprox == 2009 and mes_auth in (1, 2, 3):
        split = "eval"
      else:
        continue

      pharmacy_id = None
      for ident in receta.get("identifier") or ():
        if "medication-request-phid" in (ident.get("system") or ""):
          val = ident.get("value")
          if val is not None:
            pharmacy_id = str(val).strip()
            break

      requester_id = mapa_medicos.get(pharmacy_id) if pharmacy_id else None
      if not requester_id:
        stats["recetas_sin_medico_descartadas"] += 1
        continue

      req_id = receta.get("id")
      enc_ref = dig(receta, "encounter", "reference", default="")
      encounter_id = enc_ref.rsplit("/", 1)[-1] if "/" in enc_ref else ""

      med_ref = dig(receta, "medicationReference", "reference", default="")
      med_id = med_ref.rsplit("/", 1)[-1] if "/" in med_ref else med_ref
      medication_name = catalogo_medicamentos.get(
          med_ref
      ) or catalogo_medicamentos.get(med_id)

      if not medication_name:
        concept_txt = (
            dig(receta, "medicationCodeableConcept", "text")
            or dig(
                receta, "medicationCodeableConcept", "coding", 0, "display"
            )
            or dig(receta, "medicationCodeableConcept", "coding", 0, "code")
            or "unknown"
        )
        medication_name = normalizar_nombre_farmaco(concept_txt)

      dosage_0 = dig(receta, "dosageInstruction", 0, default={}) or {}
      dose_qty = dig(dosage_0, "doseAndRate", 0, "doseQuantity", default={}) or {}
      dose_value = dose_qty.get("value")
      dose_unit = (dose_qty.get("unit") or "").strip()

      dosage_text = dosage_0.get("text") or ""
      route_code = dig(dosage_0, "route", "coding", 0, "code", default="")
      timing_code = dig(
          dosage_0, "timing", "code", "coding", 0, "code", default=""
      )

      datos_enc = encounters.get(encounter_id) or {}
      dict_diags = diagnosticos.get(encounter_id)
      diags_texto = (
          " | ".join(f"{cod}: {txt}" for cod, txt in dict_diags.items())
          if dict_diags
          else "No diagnosis recorded"
      )

      dict_procs = procedimientos.get(encounter_id)
      procs_texto = (
          " | ".join(f"{cod}: {txt}" for cod, txt in dict_procs.items())
          if dict_procs
          else "No procedure recorded"
      )

      fila_aplanada = {
          "split": split,
          "anio_aprox": anio_aprox,
          "IdMedicationRequest": req_id,
          "authoredOn": authored_aj,
          "pharmacy_id": pharmacy_id,
          "order_provider_id": requester_id,
          "encounter_id": encounter_id,
          "encounter_class": datos_enc.get("encounter_class", "unknown"),
          "encounter_start": datos_enc.get("encounter_start", ""),
          "encounter_end": datos_enc.get("encounter_end", ""),
          "encounter_service_type": datos_enc.get(
              "encounter_service_type", "unknown"
          ),
          "encounter_diagnoses": diags_texto,
          "encounter_procedures": procs_texto,
          "patient_id": patient_id,
          "subject_id": datos_pac.get("subject_id"),
          "anchor_year": anchor_yr,
          "anchor_year_group": datos_pac.get("anchor_year_group", ""),
          "patient_gender": datos_pac.get("patient_gender", "unknown"),
          "patient_birthdate": datos_pac.get("patient_birthdate", "unknown"),
          "patient_dod": datos_pac.get("patient_dod", ""),
          "medication_id": med_id,
          "medication_name": medication_name,
          "route_code": route_code,
          "timing_code": timing_code,
          "dose_value": dose_value,
          "dose_unit": dose_unit,
          "dosage_text": dosage_text,
      }

      if split == "train":
        f_train.write(orjson.dumps(fila_aplanada) + b"\n")
        cont_train += 1
      else:
        f_eval.write(orjson.dumps(fila_aplanada) + b"\n")
        cont_eval += 1

    except Exception:
      stats["errores_procesamiento"] += 1
      continue

    if cont_total % 500_000 == 0:
      print(
          f"   Escaneadas {cont_total:,} | Train (2008): {cont_train:,} | Eval"
          f" (2009 Q1): {cont_eval:,}"
      )

os.replace(TMP_TRAIN_LOCAL, OUT_TRAIN_LOCAL)
os.replace(TMP_EVAL_LOCAL, OUT_EVAL_LOCAL)

shutil.copy2(OUT_TRAIN_LOCAL, DRIVE_TRAIN_FINAL)
shutil.copy2(OUT_EVAL_LOCAL, DRIVE_EVAL_FINAL)

print("\n🎉 ¡ETL completado y copiado a Google Drive!")
print(f"   - Train (2008): {cont_train:,} recetas -> {DRIVE_TRAIN_FINAL}")
print(f"   - Eval (2009 ene-mar): {cont_eval:,} recetas -> {DRIVE_EVAL_FINAL}")
print(f"   - Stats: {dict(stats)}")

 🚀 [B] PROCESANDO MEDICATIONREQUEST (TRAIN 2008 / EVAL ENE-MAR 2009)
   Escaneadas 1,000,000 | Train (2008): 129,024 | Eval (2009 Q1): 16,715
   Escaneadas 3,500,000 | Train (2008): 475,392 | Eval (2009 Q1): 57,922
   Escaneadas 4,000,000 | Train (2008): 540,122 | Eval (2009 Q1): 65,264
   Escaneadas 6,500,000 | Train (2008): 867,310 | Eval (2009 Q1): 103,302
   Escaneadas 10,500,000 | Train (2008): 1,411,326 | Eval (2009 Q1): 166,849

🎉 ¡ETL completado y copiado a Google Drive!
   - Train (2008): 2,074,742 recetas -> /content/drive/MyDrive/TFG/data/medication_request_2008.jsonl
   - Eval (2009 ene-mar): 243,119 recetas -> /content/drive/MyDrive/TFG/data/medication_request_2009_q1.jsonl
   - Stats: {'total_recetas_procesadas': 15416901, 'recetas_sin_medico_descartadas': 3582}


# 7. Inspección de Resultados

In [ ]:
# ==========================================
# 7. INSPECCIÓN DE RESULTADOS (TRAIN 2008 Y EVAL 2009 Q1)
# ==========================================

import json
import os
import random

ARCHIVOS_SALIDA = {
    "Entrenamiento (2008)": os.path.join(
        BASE_DRIVE, "medication_request_2008.jsonl"
    ),
    "Evaluación (2009 ene-mar)": os.path.join(
        BASE_DRIVE, "medication_request_2009_q1.jsonl"
    ),
}


def inspeccionar_jsonl(ruta, titulo, n_muestras=5, max_lineas_lectura=2000):
  print("=" * 70)
  print(f"🔍 {titulo}: {ruta}")
  print("=" * 70)

  if not os.path.exists(ruta):
    print(f"❌ No se encuentra el archivo en: {ruta}\n")
    return

  lineas_cargadas = []
  total_lineas = 0

  with open(ruta, "rt", encoding="utf-8") as f:
    for idx, linea in enumerate(f):
      total_lineas += 1
      if idx < max_lineas_lectura and linea.strip():
        lineas_cargadas.append((idx + 1, linea.strip()))

  print(f"📊 Total de líneas en el archivo: {total_lineas:,}")

  if not lineas_cargadas:
    print("⚠️ El archivo está vacío o sin líneas válidas.\n")
    return

  muestra = random.sample(
      lineas_cargadas, min(n_muestras, len(lineas_cargadas))
  )
  muestra.sort(key=lambda x: x[0])

  for i, (num_linea, contenido) in enumerate(muestra, start=1):
    try:
      datos = json.loads(contenido)
      print(f"\n--- Muestra {i} (Línea #{num_linea}) ---")
      print(json.dumps(datos, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
      print(f"\n--- Muestra {i} (Línea #{num_linea} - Error JSON) ---")
      print(contenido)
  print()


for etiqueta, ruta_archivo in ARCHIVOS_SALIDA.items():
  inspeccionar_jsonl(ruta_archivo, etiqueta, n_muestras=10)

🔍 Entrenamiento (2008): /content/drive/MyDrive/TFG/data/medication_request_2008.jsonl
📊 Total de líneas en el archivo: 2,074,742

--- Muestra 1 (Línea #360) ---
{
  "split": "train",
  "anio_aprox": 2008,
  "IdMedicationRequest": "0dbb0a55-80b8-567f-b665-f0f450bc4529",
  "authoredOn": "2008-09-29T16:15:59-04:00",
  "pharmacy_id": "18320248",
  "order_provider_id": "P93NNP",
  "encounter_id": "069637ad-7d38-5431-9137-d0a86e24a979",
  "encounter_class": "SS",
  "encounter_start": "2008-09-29T08:00:00-04:00",
  "encounter_end": "2008-10-02T17:00:00-04:00",
  "encounter_service_type": "SURG",
  "encounter_diagnoses": "2859: Anemia, unspecified | 2875: Thrombocytopenia, unspecified | 71941: Pain in joint, shoulder region | V1582: Personal history of tobacco use | 78820: Retention of urine, unspecified | 311: Depressive disorder, not elsewhere classified | 5715: Cirrhosis of liver without mention of alcohol | 07054: Chronic hepatitis C without mention of hepatic coma | 57410: Calculus of gal